## Exploration

In [ ]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_2/2019.nc"

data = xr.open_dataset(file)
print(data)

<xarray.Dataset> Size: 36GB
Dimensions:     (valid_time: 8760, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2019-01-01 ... 2019-12-31T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    expver      (valid_time) <U4 140kB ...
Data variables:
    stl2        (valid_time, latitude, longitude) float32 36GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-03T18:39 GRIB to CDM+CF via cfgrib-0.9.1...

In [7]:
import xarray as xr

# Your dataset
data = xr.open_dataset(file)

# Method 1: Select by lat/lon coordinates (easiest)
lat_target = 52.5  # Berlin latitude
lon_target = 13.4  # Berlin longitude

# Get values for both layers at that location
temp_10_40 = data['stl2'].sel(latitude=lat_target, longitude=lon_target, method='nearest')
#temp_40_100 = data['SoilTMP40_100cm_inst'].sel(lat=lat_target, lon=lon_target, method='nearest')

print(f"Location: {lat_target}°N, {lon_target}°E")
print(f"7-28cm layer: {temp_10_40.values[0]:.2f}")
#print(f"40-100cm layer: {temp_40_100.values[0]:.2f} K")

Location: 52.5°N, 13.4°E
7-28cm layer: 278.69


In [1]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/level_1/2012.nc"

data = xr.open_dataset(file)
data

KeyboardInterrupt: 

In [4]:
import xarray as xr

# Your dataset
data = xr.open_dataset(file)

# Method 1: Select by lat/lon coordinates (easiest)
lat_target = 37.2134  # Station latitude
lon_target = -98.0969  # Station longitude

# Get values for both layers at that location
temp0_10 = data['stl1'].sel(latitude=lat_target, longitude=lon_target, method='nearest')
#temp_40_100 = data['SoilTMP40_100cm_inst'].sel(lat=lat_target, lon=lon_target, method='nearest')

print(f"Location: {lat_target}°N, {lon_target}°E")
print(f"0-10cm layer: {temp0_10.values[0]:.3f}")
#print(f"40-100cm layer: {temp_40_100.values[0]:.2f} K")

Location: 37.2134°N, -98.0969°E
0-10cm layer: 289.123


In [8]:
from pathlib import Path
import xarray as xr
import pandas as pd

ERA5_L1_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/level_1"
)
year = 2020
era5_file = ERA5_L1_ROOT / f"{year}.nc"

lat_station = 37.213
lon_station = -98.097

# Open dataset
ds = xr.open_dataset(era5_file)

# Convert station lon to 0–360 to match ERA5
lon_station_360 = lon_station % 360.0

# Select nearest gridpoint (note: still latitude/longitude)
ds_sel = ds.sel(
    latitude=lat_station,
    longitude=lon_station_360,
    method="nearest",
)

print("Nearest ERA5 grid point:")
print("  latitude :", float(ds_sel["latitude"].values))
print("  longitude:", float(ds_sel["longitude"].values))

# Times of interest (match valid_time)
times = [
    "2020-12-30T20:00:00",
    "2020-12-30T21:00:00",
    "2020-12-30T22:00:00",
    "2020-12-30T23:00:00",
]

# IMPORTANT: use 'valid_time' not 'time'
stl1_sel = ds_sel["stl1"].sel(valid_time=times)

# Convert to a small DataFrame
df = stl1_sel.to_series().reset_index()
df.rename(columns={"valid_time": "datetime", "stl1": "stl1_K"}, inplace=True)

print("\nRaw ERA5 stl1 values at nearest grid point:")
print(df.to_string(index=False))

ds.close()


Nearest ERA5 grid point:
  latitude : 37.25
  longitude: 262.0

Raw ERA5 stl1 values at nearest grid point:
           datetime     stl1_K
2020-12-30 20:00:00 279.319916
2020-12-30 21:00:00 279.965698
2020-12-30 22:00:00 279.029541
2020-12-30 23:00:00 278.706757


## Test: Combining era5 and in-situ 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
INSITU_ARM_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/in_situ/combine/ARM"
)
ERA5_L2_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_2"
)
ERA5_L3_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_3"
)
OUT_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Helper: load ERA5 time series at nearest grid point for one station
# ------------------------------------------------------------------
def extract_era5_for_station(lat_station, lon_station, years):
    """
    Returns:
      era5_df: DataFrame with index 'datetime' and columns
               ['ts_era5_l2', 'ts_era5_l3', 'lat_era5', 'lon_era5'].
               lat_era5/lon_era5 are constant columns, lon_era5 in -180..180.
    """
    # ERA5 longitudes are 0..360, station lon is likely -180..180
    lon_station_360 = lon_station % 360.0

    dfs_year = []
    lat_era5_val = None
    lon_era5_val = None

    for year in sorted(set(years)):
        f2 = ERA5_L2_ROOT / f"{year}.nc"
        f3 = ERA5_L3_ROOT / f"{year}.nc"
        if not f2.exists() or not f3.exists():
            print(f"  ERA5 file missing for year {year}, skipping this year.")
            continue

        ds2 = xr.open_dataset(f2)
        ds3 = xr.open_dataset(f3)

        # nearest grid point
        ds2_sel = ds2.sel(latitude=lat_station, longitude=lon_station_360, method="nearest")
        ds3_sel = ds3.sel(latitude=lat_station, longitude=lon_station_360, method="nearest")

        # actual ERA5 coordinates (convert lon from 0..360 to -180..180)
        lat_era = float(ds2_sel["latitude"].values)
        lon_era_raw = float(ds2_sel["longitude"].values)
        lon_era = ((lon_era_raw + 180.0) % 360.0) - 180.0

        lat_era5_val = lat_era if lat_era5_val is None else lat_era5_val
        lon_era5_val = lon_era if lon_era5_val is None else lon_era5_val

        t2 = ds2_sel["stl2"].to_series()
        t3 = ds3_sel["stl3"].to_series()

        df_y = pd.DataFrame(
            {
                "ts_era5_l2": t2,
                "ts_era5_l3": t3,
            }
        )
        df_y.index.name = "datetime"
        dfs_year.append(df_y)

        ds2.close()
        ds3.close()

    if not dfs_year:
        return None

    era5_df = pd.concat(dfs_year).sort_index()
    era5_df["lat_era5"] = lat_era5_val
    era5_df["lon_era5"] = lon_era5_val
    return era5_df

# ------------------------------------------------------------------
# Process each ARM station
# ------------------------------------------------------------------
for station_dir in sorted(INSITU_ARM_ROOT.iterdir()):
    if not station_dir.is_dir():
        continue

    station_files = list(station_dir.glob("*_soil_temperature_depths.csv"))
    if not station_files:
        print(f"No in-situ CSV for station {station_dir.name}, skipping.")
        continue

    csv_path = station_files[0]
    print(f"\nProcessing ARM / {station_dir.name} -> {csv_path.name}")

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  Failed to read {csv_path}: {e}")
        continue

    needed = [
        "date", "time",
        "ts_station_k",
        "lat", "lon",
        "elev", "cc", "lc",
        "land_cover_group", "climate_group",
        "temp_class", "elevation_class",
    ]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"  Missing columns {missing}, skipping station.")
        continue

    # build datetime
    df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
    if df["datetime"].isna().all():
        print("  All datetimes are NaT, skipping station.")
        continue

    df = df.dropna(subset=["datetime"]).sort_values("datetime")
    df = df.set_index("datetime")

    lat_station = float(df["lat"].iloc[0])
    lon_station = float(df["lon"].iloc[0])
    years = df.index.year.unique()

    era5_df = extract_era5_for_station(lat_station, lon_station, years)
    if era5_df is None:
        print("  No ERA5 data found for this station, skipping.")
        continue

    combined = df.join(era5_df, how="inner")
    if combined.empty:
        print("  No overlapping timestamps between station and ERA5, skipping.")
        continue

    # reset index to date/time
    combined = combined.reset_index()
    combined["date"] = combined["datetime"].dt.strftime("%Y-%m-%d")
    combined["time"] = combined["datetime"].dt.strftime("%H:%M")
    combined = combined.drop(columns=["datetime"])

    # ------------------------------------------------------------------
    # Drop ts_station and all T_* layer columns (you said they are not needed here)
    # ------------------------------------------------------------------
    drop_cols = []
    if "ts_station" in combined.columns:
        drop_cols.append("ts_station")
    drop_cols.extend([c for c in combined.columns if c.startswith("T_")])

    if drop_cols:
        combined = combined.drop(columns=drop_cols)

    # ------------------------------------------------------------------
    # Column ordering
    # ------------------------------------------------------------------
    meta_cols = [
        "date", "time",
        "ts_station_k",
        "lat", "lon",
        "lat_era5", "lon_era5",
        "elev", "cc", "lc",
        "land_cover_group", "climate_group",
        "temp_class", "elevation_class",
    ]
    era5_cols = ["ts_era5_l2", "ts_era5_l3"]

    other_cols = [
        c for c in combined.columns
        if c not in meta_cols + era5_cols
    ]

    final_cols = meta_cols + other_cols + era5_cols
    final_cols = [c for c in final_cols if c in combined.columns]
    combined = combined[final_cols]

    out_station_dir = OUT_ROOT / station_dir.name
    out_station_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_station_dir / f"{station_dir.name}_insitu_era5_soil_temperature.csv"
    combined.to_csv(out_csv, index=False, float_format="%.3f")

    print(f"  Wrote: {out_csv}")



Processing ARM / Anthony -> ARM_Anthony_soil_temperature_depths.csv
  Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Anthony/Anthony_insitu_era5_soil_temperature.csv

Processing ARM / Ashton -> ARM_Ashton_soil_temperature_depths.csv
  Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Ashton/Ashton_insitu_era5_soil_temperature.csv

Processing ARM / Byron -> ARM_Byron_soil_temperature_depths.csv
  Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Byron/Byron_insitu_era5_soil_temperature.csv

Processing ARM / Lamont-CF1 -> ARM_Lamont-CF1_soil_temperature_depths.csv
  Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Lamont-CF1/Lamont-CF1_insitu_era5_soil_temperature.csv

Processing ARM / MapleCity -> ARM_MapleCity_soil_temperature_depths.csv
  Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/MapleCity/MapleCity_insit

## Combining all

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
INSITU_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/in_situ/combine"
)
ERA5_L2_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_2"
)
ERA5_L3_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_3"
)
OUT_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Helper: load ERA5 time series at nearest grid point for one station
# ------------------------------------------------------------------
def extract_era5_for_station(lat_station, lon_station, years):
    """
    Returns:
      era5_df: DataFrame with index 'datetime' and columns
               ['ts_era5_l2', 'ts_era5_l3', 'lat_era5', 'lon_era5'].
               lat_era5/lon_era5 are constant columns, lon_era5 in -180..180.
    """
    lon_station_360 = lon_station % 360.0

    dfs_year = []
    lat_era5_val = None
    lon_era5_val = None

    for year in sorted(set(years)):
        f2 = ERA5_L2_ROOT / f"{year}.nc"
        f3 = ERA5_L3_ROOT / f"{year}.nc"
        if not f2.exists() or not f3.exists():
            print(f"  ERA5 file missing for year {year}, skipping this year.")
            continue

        ds2 = xr.open_dataset(f2)
        ds3 = xr.open_dataset(f3)

        ds2_sel = ds2.sel(latitude=lat_station, longitude=lon_station_360, method="nearest")
        ds3_sel = ds3.sel(latitude=lat_station, longitude=lon_station_360, method="nearest")

        lat_era = float(ds2_sel["latitude"].values)
        lon_era_raw = float(ds2_sel["longitude"].values)
        lon_era = ((lon_era_raw + 180.0) % 360.0) - 180.0

        lat_era5_val = lat_era if lat_era5_val is None else lat_era5_val
        lon_era5_val = lon_era if lon_era5_val is None else lon_era5_val

        t2 = ds2_sel["stl2"].to_series()
        t3 = ds3_sel["stl3"].to_series()

        df_y = pd.DataFrame(
            {
                "ts_era5_l2": t2,
                "ts_era5_l3": t3,
            }
        )
        df_y.index.name = "datetime"
        dfs_year.append(df_y)

        ds2.close()
        ds3.close()

    if not dfs_year:
        return None

    era5_df = pd.concat(dfs_year).sort_index()
    era5_df["lat_era5"] = lat_era5_val
    era5_df["lon_era5"] = lon_era5_val
    return era5_df


# ------------------------------------------------------------------
# Process each network and station
# ------------------------------------------------------------------
for network_dir in sorted(INSITU_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net_id = network_dir.name
    print(f"\n=== Network: {net_id} ===")

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue

        st_id = station_dir.name
        station_files = list(station_dir.glob("*_soil_temperature_depths.csv"))
        if not station_files:
            print(f"  {net_id}/{st_id}: no in-situ CSV, skipping.")
            continue

        csv_path = station_files[0]
        print(f"  Processing {net_id} / {st_id} -> {csv_path.name}")

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"    Failed to read {csv_path}: {e}")
            continue

        needed = [
            "date", "time",
            "ts_station_k",
            "lat", "lon",
            "elev", "cc", "lc",
            "land_cover_group", "climate_group",
            "temp_class", "elevation_class",
        ]
        missing = [c for c in needed if c not in df.columns]
        if missing:
            print(f"    Missing columns {missing}, skipping station.")
            continue

        # NEW: ensure ts_station_k is rounded to 3 decimals
        df["ts_station_k"] = df["ts_station_k"].round(3)

        # build datetime
        df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
        if df["datetime"].isna().all():
            print("    All datetimes are NaT, skipping station.")
            continue

        df = df.dropna(subset=["datetime"]).sort_values("datetime")
        df = df.set_index("datetime")

        lat_station = float(df["lat"].iloc[0])
        lon_station = float(df["lon"].iloc[0])
        years = df.index.year.unique()

        era5_df = extract_era5_for_station(lat_station, lon_station, years)
        if era5_df is None:
            print("    No ERA5 data found for this station, skipping.")
            continue

        combined = df.join(era5_df, how="inner")
        if combined.empty:
            print("    No overlapping timestamps between station and ERA5, skipping.")
            continue

        # strict paired availability
        mask_bad = (
            combined["ts_station_k"].isna()
            | combined["ts_era5_l2"].isna()
            | combined["ts_era5_l3"].isna()
        )
        combined.loc[mask_bad, ["ts_station_k", "ts_era5_l2", "ts_era5_l3"]] = np.nan

        combined = combined.reset_index()
        combined["date"] = combined["datetime"].dt.strftime("%Y-%m-%d")
        combined["time"] = combined["datetime"].dt.strftime("%H:%M")
        combined = combined.drop(columns=["datetime"])

        # drop ts_station and all T_* layer columns
        drop_cols = []
        if "ts_station" in combined.columns:
            drop_cols.append("ts_station")
        drop_cols.extend([c for c in combined.columns if c.startswith("T_")])
        if drop_cols:
            combined = combined.drop(columns=drop_cols)

        # column ordering
        meta_cols = [
            "date", "time",
            "ts_station_k",
            "lat", "lon",
            "lat_era5", "lon_era5",
            "elev", "cc", "lc",
            "land_cover_group", "climate_group",
            "temp_class", "elevation_class",
        ]
        era5_cols = ["ts_era5_l2", "ts_era5_l3"]

        other_cols = [
            c for c in combined.columns
            if c not in meta_cols + era5_cols
        ]

        final_cols = meta_cols + other_cols + era5_cols
        final_cols = [c for c in final_cols if c in combined.columns]
        combined = combined[final_cols]

        out_station_dir = OUT_ROOT / net_id / st_id
        out_station_dir.mkdir(parents=True, exist_ok=True)

        out_csv = out_station_dir / f"{st_id}_insitu_era5_soil_temperature.csv"
        combined.to_csv(out_csv, index=False, float_format="%.3f")

        print(f"    Wrote: {out_csv}")



=== Network: ARM ===
  Processing ARM / Anthony -> Anthony_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Anthony/Anthony_insitu_era5_soil_temperature.csv
  Processing ARM / Ashton -> Ashton_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Ashton/Ashton_insitu_era5_soil_temperature.csv
  Processing ARM / Byron -> Byron_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Byron/Byron_insitu_era5_soil_temperature.csv
  Processing ARM / Lamont-CF1 -> Lamont-CF1_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/Lamont-CF1/Lamont-CF1_insitu_era5_soil_temperature.csv
  Processing ARM / MapleCity -> MapleCity_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ARM/MapleCit

## Test: Computing the root zone depth weighted mean, and adding new column as ts_era5

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# Input: only BNZ-LTER combined files
BNZ_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/BNZ-LTER"
)

# Output test path
OUT_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ERA5 layer thicknesses within 0.10–1.00 m
d2 = 0.28 - 0.10  # 0.18 m
d3 = 1.00 - 0.28  # 0.72 m
den = d2 + d3     # 0.90 m

for csv in BNZ_ROOT.rglob("*.csv"):
    df = pd.read_csv(csv)

    if "ts_era5_l2" not in df.columns or "ts_era5_l3" not in df.columns:
        print(f"Skipping (missing ERA5 cols): {csv}")
        continue
    if "ts_station_k" not in df.columns:
        print(f"Skipping (no ts_station_k): {csv}")
        continue

    # weighted mean (Kelvin)
    num = df["ts_era5_l2"] * d2 + df["ts_era5_l3"] * d3
    ts_era5 = num / den

    # if either level is NaN -> NaN
    mask_bad = df["ts_era5_l2"].isna() | df["ts_era5_l3"].isna()
    ts_era5[mask_bad] = np.nan

    # round to 3 decimals
    ts_era5 = ts_era5.round(3)

    # drop any old ts_era5
    if "ts_era5" in df.columns:
        df = df.drop(columns=["ts_era5"])

    # insert after ts_station_k
    idx = df.columns.get_loc("ts_station_k")
    df.insert(idx + 1, "ts_era5", ts_era5)

    # write to test2, preserving subfolder structure and filename
    rel_path = csv.relative_to(BNZ_ROOT)          # station/filename.csv
    out_path = OUT_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(out_path, index=False, float_format="%.3f")
    print(f"Wrote test file: {out_path}")


Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/UP2A/UP2A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/FP2A/FP2A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/LTER1/LTER1_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/FP4A/FP4A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/UP1A/UP1A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/FP5A/FP5A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/test2/FP3A/FP3A_insitu_era5_soil_temperature.csv
Wrote test file: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5

## Applying root zone depth weighted mean to all networks

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine")

# ERA5 layer thicknesses within 0.10–1.00 m
d2 = 0.28 - 0.10  # 0.18 m
d3 = 1.00 - 0.28  # 0.72 m
den = d2 + d3     # 0.90 m

for csv in ROOT.rglob("*.csv"):
    df = pd.read_csv(csv)

    if "ts_era5_l2" not in df.columns or "ts_era5_l3" not in df.columns:
        print(f"Skipping (missing ERA5 cols): {csv}")
        continue
    if "ts_station_k" not in df.columns:
        print(f"Skipping (no ts_station_k): {csv}")
        continue

    # weighted mean (Kelvin)
    num = df["ts_era5_l2"] * d2 + df["ts_era5_l3"] * d3
    ts_era5 = num / den

    # if either level is NaN -> NaN
    mask_bad = df["ts_era5_l2"].isna() | df["ts_era5_l3"].isna()
    ts_era5[mask_bad] = np.nan

    # round to 3 decimals
    ts_era5 = ts_era5.round(3)

    # drop any old ts_era5
    if "ts_era5" in df.columns:
        df = df.drop(columns=["ts_era5"])

    # insert after ts_station_k
    idx = df.columns.get_loc("ts_station_k")
    df.insert(idx + 1, "ts_era5", ts_era5)

    # overwrite same file
    df.to_csv(csv, index=False, float_format="%.3f")
    print(f"Updated: {csv}")


Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ROMPS/AbramovGlacier/AbramovGlacier_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ROMPS/GolubinGlacier/GolubinGlacier_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ROMPS/Baytik/Baytik_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ROMPS/Asai/Asai_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/ROMPS/Ayvadzh/Ayvadzh_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/HOBE/3.04/3.04_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/combine/HOBE/2.11/2.11_insitu_era5_soil_temperature.csv
Updated: /home/aminr/Desktop/Master

## Checking and comparing era5_insitu vs raw era5 data

- compare the combined era5_insitu with raw era5 data at specific location and time using the raw data file path.

- To check if the combine data is correctly extracted for that station coordinates

- After checking multiple times for both layers, it was conformed that the data was correctly extracted, and it was safe to delete the raw files from local machine, but it is still available on the cluster

In [14]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/era5/level_2/2012.nc"

data = xr.open_dataset(file)
data

<xarray.Dataset> Size: 36GB
Dimensions:     (valid_time: 8784, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2012-01-01 ... 2012-12-31T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    expver      (valid_time) <U4 141kB ...
Data variables:
    stl2        (valid_time, latitude, longitude) float32 36GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-03T14:23 GRIB to CDM+CF via cfgrib-0.9.1...

In [15]:
import xarray as xr

data = xr.open_dataset(file)
lat_target = 39.750
lon_target = 71.500  # Convert to 0-360° format = 71.586

# Select specific time
temp_10_40 = data['stl2'].sel(
    valid_time="2012-1-1T20:00:00",  # Use valid_time and 2012 year
    latitude=lat_target, 
    longitude=lon_target, 
    method='nearest'
)

print(f"Location: {lat_target}°N, {lon_target}°E")
print(f"7-28cm layer: {temp_10_40.values:.3f} K")

Location: 39.75°N, 71.5°E
7-28cm layer: 271.726 K


## computing the standard deviation and correlation between the station and product

- using combine file (era5_in_situ)

In [4]:
import pandas as pd
from pathlib import Path

# Root dirs
ERA5_COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine")
OUT_SUMMARY = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/era5_insitu_sd_corr_summary.csv")

results = []

for network_dir in sorted(ERA5_COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    network = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        station = station_dir.name

        csv_files = list(station_dir.glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV in {station_dir}, skipping.")
            continue

        csv_path = csv_files[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[ERROR] Cannot read {csv_path}: {e}")
            results.append({
                "network": network,
                "station": station,
                "n_points": 0,
                "sd_insitu": None,
                "sd_era5": None,
                "sd_ratio_era5_insitu": None,
                "corr_era5_insitu": None,
                "pass_sd_filter": False,
                "pass_corr_filter": False,
            })
            continue

        if "date" in df.columns and "time" in df.columns:
            df["datetime"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce"
            )

        if not {"ts_station_k", "ts_era5"}.issubset(df.columns):
            print(f"[WARN] Missing ts_station_k or ts_era5 in {csv_path}, skipping station.")
            continue

        sub = df[["ts_station_k", "ts_era5"]].dropna()
        n_points = len(sub)

        if n_points < 2:
            print(f"[WARN] Not enough valid points for {network}/{station} ({n_points}), skipping stats.")
            sd_insitu = sd_era5 = ratio = corr = None
            pass_sd = pass_corr = False
        else:
            sd_insitu = sub["ts_station_k"].std()
            sd_era5 = sub["ts_era5"].std()
            ratio = sd_era5 / sd_insitu if sd_insitu and sd_insitu > 0 else None
            corr = sub["ts_station_k"].corr(sub["ts_era5"])

            if ratio is not None and 0.1 <= ratio <= 3:
                pass_sd = True
            else:
                pass_sd = False

            pass_corr = corr is not None and corr >= 0.5

        results.append({
            "network": network,
            "station": station,
            "n_points": n_points,
            "sd_insitu": sd_insitu,
            "sd_era5": sd_era5,
            "sd_ratio_era5_insitu": ratio,
            "corr_era5_insitu": corr,
            "pass_sd_filter": pass_sd,
            "pass_corr_filter": pass_corr,
        })

# Build DataFrame
summary_df = pd.DataFrame(results)

# Round numeric columns to 3 decimals
num_cols = ["sd_insitu", "sd_era5", "sd_ratio_era5_insitu", "corr_era5_insitu"]
for col in num_cols:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].round(3)

# Save summary (without csv_path)
summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved summary to {OUT_SUMMARY}")


Saved summary to /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/era5_insitu_sd_corr_summary.csv


## Checking the summary of the pass/fail staions

In [7]:
import pandas as pd
from pathlib import Path

summary_path = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/era5_insitu_sd_corr_summary.csv")

df = pd.read_csv(summary_path)

# Total stations
total = len(df)

# Stations that pass both SD and correlation filters
pass_both = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]

# Stations failing at least one filter
fail_any = df[(df["pass_sd_filter"] == False) | (df["pass_corr_filter"] == False)]

print(f"Total stations          : {total}")
print(f"Pass both (SD & corr)   : {len(pass_both)}")
print(f"Fail SD or corr (or both): {len(fail_any)}")

# Optional: counts per network
print("\nPer network (pass both):")
print(pass_both.groupby("network")["station"].count())

print("\nPer network (fail any):")
print(fail_any.groupby("network")["station"].count())

# NEW: print failing station names per network
print("\nFailing stations by network:")
for network, sub in fail_any.groupby("network"):
    station_list = sorted(sub["station"].unique())
    print(f"{network}: {station_list} = {len(station_list)}")


Total stations          : 1173
Pass both (SD & corr)   : 1153
Fail SD or corr (or both): 20

Per network (pass both):
network
ARM                   14
BIEBRZA_S-1           18
BNZ-LTER               8
Berlin                 1
COSMOS-UK             49
CTP_SMTMN             57
CW3E                   8
DAHRA                  1
FLUXNET-AMERIFLUX      5
FMI                   10
FR_Aqui                5
HOAL                  29
HOBE                  30
LABFLUX                1
MAQU                  16
MOL-RAO                2
NAQU                  10
NGARI                 20
ORACLE                 1
RISMA                 23
ROMPS                  5
Ru_CFR                 2
SCAN                 204
SKKU                   1
SMN-SDR               33
SMOSMANIA             22
SNOTEL               434
STEMS                  2
TERENO                 5
TWENTE                25
USCRN                 93
WEGENERNET            11
XMS-CAT                8
Name: station, dtype: int64

Per network (fail an

## Taking pixel mean

In [1]:
import os
import glob
import pandas as pd
from collections import Counter
from pathlib import Path

# Root of ERA5–in situ combined files
COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine"

# Output root for pixel-mean processing
OUT_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean"

# Summary file with filters (already created)
SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/era5_insitu_sd_corr_summary.csv"


def majority(series):
    """Most common non-NaN value in a series."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return Counter(vals).most_common(1)[0][0]


def classify_elevation(elev):
    """Classify elevation using DEM-I/II/III/IV thresholds."""
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"

    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"


def load_pass_stations(summary_path):
    """
    Read summary CSV and return a set of (network, station) pairs
    that passed BOTH SD and correlation filters.
    """
    df = pd.read_csv(summary_path)
    ok = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]
    return set(zip(ok["network"], ok["station"]))


def process_network(network, pass_stations):
    base_in = os.path.join(COMBINE_ROOT, network)
    base_out = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)  # only base

    print(f"\n=== Network: {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    all_records = []
    used_station_dirs = []

    # Load station data, but only for stations that passed filters
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        # Skip station if it failed any filter in summary
        if (network, station_name) not in pass_stations:
            print(f"  Skipping {network}/{station_name} (failed filter in summary).")
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            df = pd.read_csv(f)

            # Sanity check: required columns
            required_cols = {"date", "time", "ts_station_k", "ts_era5", "lat_era5", "lon_era5"}
            if not required_cols.issubset(df.columns):
                print(f"  [WARN] Missing required columns in {f}, skipping this file.")
                continue

            df["station"] = station_name
            all_records.append(df)
            used_station_dirs.append(station_dir)

    if not all_records:
        print(f"No valid CSV files found under {base_in}/* (after filtering), skipping.")
        return

    df_all = pd.concat(all_records, ignore_index=True)
    print(f"Loaded {len(set(used_station_dirs))} stations after filtering, total rows: {len(df_all)}")

    # DENSE PIXELS: at least 2 stations in same (lat_era5, lon_era5)
    pixel_station_counts = (
        df_all.groupby(["lat_era5", "lon_era5"])["station"]
              .nunique()
              .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_era5", "lon_era5"]]
    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    n_sparse_files = 0
    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # DENSE
    print("Processing dense pixels (pixel-mean files)...")
    df_dense_all = df_all.merge(dense_pixels, on=["lat_era5", "lon_era5"], how="inner")
    print("Rows in dense pixels (before grouping):", len(df_dense_all))

    stations_in_dense = set()

    # NEW: also read existing dense files to populate stations_in_dense
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    if not df_dense_all.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_era5"]
            plon = pix["lon_era5"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            # NEW: skip this pixel if its dense file already exists
            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_era5={plat}, lon_era5={plon} (file exists).")
                continue

            df_pix = df_dense_all[
                (df_dense_all["lat_era5"] == plat) &
                (df_dense_all["lon_era5"] == plon)
            ].copy()

            if df_pix.empty:
                continue

            print(f"  Dense pixel at lat_era5={plat}, lon_era5={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                # Only consider rows with valid in-situ value
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref = g_valid["ts_station_k"].mean()
                    # For a given pixel and time, ts_era5 should be same across stations
                    ts_era5 = g_valid["ts_era5"].iloc[0]

                    lat_val = majority(g_valid["lat"])
                    lon_val = majority(g_valid["lon"])
                    cc_val = majority(g_valid["cc"])
                    lc_val = majority(g_valid["lc"])
                    lcg_val = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    # elevation: mean of station elevations
                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref = pd.NA
                    ts_era5 = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_era5": ts_era5,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_era5"] = pd.to_numeric(df_pix_mean["ts_era5"], errors="coerce").round(3)

            # retain pixel coordinates and station list
            df_pix_mean["lat_era5"] = plat
            df_pix_mean["lon_era5"] = plon
            df_pix_mean["stations"] = stations_str

            # out_dense already defined above
            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")
    else:
        print("No dense pixels found to process.")

    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")

    # SPARSE: station-wise files, excluding stations used in any dense pixel
    print("Writing sparse (station-wise) files (excluding dense stations)...")
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        # Skip if station failed filters globally
        if (network, station_name) not in pass_stations:
            continue

        # Skip if this station was already included in any dense pixel (existing or new)
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            # NEW: skip if sparse file already exists
            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            df = pd.read_csv(f)

            # Rename in-situ and ERA5 columns to match dense output
            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            if "ts_era5" in df.columns:
                rename_map["ts_era5"] = "ts_era5"  # same name, but kept for symmetry

            df = df.rename(columns=rename_map)
            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Stations found (raw): {n_stations_found}")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


if __name__ == "__main__":
    # Load list of stations that passed ERA5 SD & correlation filters
    pass_stations = load_pass_stations(SUMMARY_PATH)

    # Discover networks under COMBINE_ROOT
    network_dirs = sorted(
        d for d in glob.glob(os.path.join(COMBINE_ROOT, "*"))
        if os.path.isdir(d)
    )
    networks = [os.path.basename(d) for d in network_dirs]

    print("Networks found:", networks)
    for net in networks:
        process_network(net, pass_stations)


Networks found: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'LABFLUX', 'MAQU', 'MOL-RAO', 'NAQU', 'NGARI', 'ORACLE', 'OZNET', 'RISMA', 'ROMPS', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'STEMS', 'TERENO', 'TWENTE', 'USCRN', 'WEGENERNET', 'XMS-CAT']

=== Network: ARM ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine/ARM
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/ARM
Found 14 station directories.
Loaded 14 stations after filtering, total rows: 776154
Total unique pixels: 14
Dense pixels (n_stations >= 2): 0
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 0
No dense pixels found to process.
Stations that appear in dense pixels: 0
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for ARM/Anthony (/home/aminr/Desk

/tmp/ipykernel_233915/929750714.py:134: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


Loaded 57 stations after filtering, total rows: 2248630
Total unique pixels: 13
Dense pixels (n_stations >= 2): 12
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 2207262


/tmp/ipykernel_233915/929750714.py:134: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)
/tmp/ipykernel_233915/929750714.py:134: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_era5=31.0, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.0, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.0 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=92.0 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=92.5 (file exists).
  Skipping dense pixel lat_era5=32.0, lon_era5=91.75 (file exists).
Stations that appear in dense pixels: 56
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for CTP_SMTMN/M15 (/home/aminr/Desktop/Master_Thesi

...

  Skipping dense pixel lat_era5=52.25, lon_era5=14.0 (file exists).
Stations that appear in dense pixels: 2
Writing sparse (station-wise) files (excluding dense stations)...
Finished sparse files. Processed 0 station CSV files (excluding 2 dense stations).
All done for MOL-RAO.
===== SUMMARY =====
Stations found (raw): 2
Dense pixels processed (with valid data): 0
Dense pixel files written: 0

=== Network: NAQU ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine/NAQU
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/NAQU
Found 10 station directories.
Loaded 10 stations after filtering, total rows: 424919
Total unique pixels: 3
Dense pixels (n_stations >= 2): 2
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 357455


/tmp/ipykernel_233915/929750714.py:134: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_era5=31.25, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.0 (file exists).
Stations that appear in dense pixels: 9
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for NAQU/NQNorth (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/NAQU/sparse/NQNorth_insitu_era5_soil_temperature.csv exists).
Finished sparse files. Processed 0 station CSV files (excluding 9 dense stations).
All done for NAQU.
===== SUMMARY =====
Stations found (raw): 10
Dense pixels processed (with valid data): 0
Dense pixel files written: 0

=== Network: NGARI ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine/NGARI
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/NGARI
Found 20 station directories.
Loaded 20 stations after filtering, total rows: 947285
Total unique pixels: 6
Dense pixels (n_stations >= 2):

/tmp/ipykernel_233915/929750714.py:134: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_era5=20.0, lon_era5=-155.5 (file exists).
  Skipping dense pixel lat_era5=35.0, lon_era5=-119.5 (file exists).
  Skipping dense pixel lat_era5=35.0, lon_era5=-86.5 (file exists).
  Skipping dense pixel lat_era5=36.25, lon_era5=-115.75 (file exists).
  Skipping dense pixel lat_era5=36.25, lon_era5=-115.5 (file exists).
  Skipping dense pixel lat_era5=37.75, lon_era5=-109.25 (file exists).
  Skipping dense pixel lat_era5=38.5, lon_era5=-92.25 (file exists).
  Skipping dense pixel lat_era5=40.75, lon_era5=-104.75 (file exists).
  Skipping dense pixel lat_era5=41.25, lon_era5=-111.25 (file exists).
Stations that appear in dense pixels: 21
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for SCAN/AAMU-JTG (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/SCAN/sparse/AAMU-JTG_insitu_era5_soil_temperature.csv exists).
  Skipping sparse file for SCAN/Abrams (/home/aminr/Desktop/Master_Thesis/Glo

### 1. Pixel mean files validation: 

In [2]:
import pandas as pd
from pathlib import Path
import glob
import os

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/era5_insitu_sd_corr_summary.csv")

summary = pd.read_csv(SUMMARY_PATH)

# Stations that passed both filters
passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

# Stations that failed any filter
failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

dense_stations = set()
sparse_stations = set()

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    net_out = OUT_ROOT / network
    dense_dir = net_out / "dense"
    sparse_dir = net_out / "sparse"

    # collect from dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            df = pd.read_csv(f)
            if "stations" in df.columns and not df.empty:
                for st in str(df["stations"].iloc[0]).split(","):
                    st = st.strip()
                    if st:
                        dense_stations.add((network, st))

    # collect from sparse
    if sparse_dir.exists():
        for st_dir in sorted((COMBINE_ROOT / network).iterdir()):
            if not st_dir.is_dir():
                continue
            station = st_dir.name
            pattern = str(sparse_dir / f"{station}*.csv")
            files = glob.glob(pattern)
            if files:
                sparse_stations.add((network, station))

print("Total passed stations        :", len(passed_set))
print("Stations in dense pixels     :", len(dense_stations))
print("Stations in sparse outputs   :", len(sparse_stations))

print("\nStations that passed but not in dense or sparse:")
missing = passed_set - dense_stations - sparse_stations
print(missing)

print("\nStations that failed but appear in dense or sparse (should be empty):")
bad = (failed_set & dense_stations) | (failed_set & sparse_stations)
print(bad)


/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/590355681.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read

Total passed stations        : 1153
Stations in dense pixels     : 518
Stations in sparse outputs   : 637

Stations that passed but not in dense or sparse:
set()

Stations that failed but appear in dense or sparse (should be empty):
set()


###  2. Check dense files: ts_reference and metadata

In [1]:
import pandas as pd
from pathlib import Path
import glob
import numpy as np

NETWORK = "BIEBRZA_S-1"  # change to other networks as needed

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine") / NETWORK
DENSE_DIR    = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean") / NETWORK / "dense"

def majority(series):
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return max(set(vals), key=vals.count)

def classify_elevation(elev):
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"
    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"

dense_files = glob.glob(str(DENSE_DIR / f"{NETWORK}_dense_lat*_lon*.csv"))
print(f"Found {len(dense_files)} dense files for {NETWORK}")

mismatches = []

for f_dense in dense_files:
    df_dense = pd.read_csv(f_dense)
    if df_dense.empty:
        continue

    stations = str(df_dense["stations"].iloc[0]).split(",")
    stations = [s.strip() for s in stations if s.strip()]

    # load original station data for these stations
    all_records = []
    for st in stations:
        st_dir = COMBINE_ROOT / st
        for f in st_dir.glob("*.csv"):
            df = pd.read_csv(f)
            df["station"] = st
            all_records.append(df)

    if not all_records:
        continue

    df_all = pd.concat(all_records, ignore_index=True)

    # pixel coordinate from dense file
    plat = df_dense["lat_era5"].iloc[0]
    plon = df_dense["lon_era5"].iloc[0]
    df_pix = df_all[(df_all["lat_era5"] == plat) & (df_all["lon_era5"] == plon)].copy()

    # merge on date+time to compare row-by-row
    df_pix["key"] = df_pix["date"].astype(str) + " " + df_pix["time"].astype(str)
    df_dense["key"] = df_dense["date"].astype(str) + " " + df_dense["time"].astype(str)

    for key, g in df_pix.groupby("key"):
        dense_row = df_dense[df_dense["key"] == key]
        if dense_row.empty:
            continue

        dense_row = dense_row.iloc[0]

        g_valid = g[g["ts_station_k"].notna()]
        n_valid = g_valid["station"].nunique()

        # recompute reference and metadata
        if n_valid >= 2:
            ts_ref_true = round(g_valid["ts_station_k"].mean(), 3)
            ts_era5_true = g_valid["ts_era5"].iloc[0]

            lat_true  = majority(g_valid["lat"])
            lon_true  = majority(g_valid["lon"])
            cc_true   = majority(g_valid["cc"])
            lc_true   = majority(g_valid["lc"])
            lcg_true  = majority(g_valid["land_cover_group"])
            clim_true = majority(g_valid["climate_group"])
            tcls_true = majority(g_valid["temp_class"])
            elev_mean_true = g_valid["elev"].mean()
            elevc_true = classify_elevation(elev_mean_true)
        else:
            ts_ref_true = np.nan
            ts_era5_true = np.nan
            lat_true = lon_true = cc_true = lc_true = lcg_true = clim_true = tcls_true = np.nan
            elev_mean_true = np.nan
            elevc_true = "DEM-Unknown"

        # compare with dense file
        def neq(a, b, tol=1e-3):
            if pd.isna(a) and pd.isna(b):
                return False
            if isinstance(a, (float, int)) and isinstance(b, (float, int)):
                return abs(a - b) > tol
            return a != b

        issues = []

        if neq(dense_row["ts_reference"], ts_ref_true):
            issues.append(f"ts_reference {dense_row['ts_reference']} vs {ts_ref_true}")
        if neq(dense_row["ts_era5"], ts_era5_true):
            issues.append(f"ts_era5 {dense_row['ts_era5']} vs {ts_era5_true}")
        if neq(dense_row["lat"], lat_true):
            issues.append(f"lat {dense_row['lat']} vs {lat_true}")
        if neq(dense_row["lon"], lon_true):
            issues.append(f"lon {dense_row['lon']} vs {lon_true}")
        if neq(dense_row["cc"], cc_true):
            issues.append(f"cc {dense_row['cc']} vs {cc_true}")
        if neq(dense_row["lc"], lc_true):
            issues.append(f"lc {dense_row['lc']} vs {lc_true}")
        if neq(dense_row["land_cover_group"], lcg_true):
            issues.append(f"land_cover_group {dense_row['land_cover_group']} vs {lcg_true}")
        if neq(dense_row["climate_group"], clim_true):
            issues.append(f"climate_group {dense_row['climate_group']} vs {clim_true}")
        if neq(dense_row["temp_class"], tcls_true):
            issues.append(f"temp_class {dense_row['temp_class']} vs {tcls_true}")
        if neq(dense_row["elev"], elev_mean_true):
            issues.append(f"elev {dense_row['elev']} vs {elev_mean_true}")
        if neq(dense_row["elevation_class"], elevc_true):
            issues.append(f"elevation_class {dense_row['elevation_class']} vs {elevc_true}")

        if issues:
            mismatches.append((f_dense, key, issues))

print(f"\nTotal mismatches found: {len(mismatches)}")
for f_dense, key, issues in mismatches[:20]:
    print(f"Dense file: {f_dense}, datetime: {key}")
    for issue in issues:
        print("  -", issue)


Found 2 dense files for BIEBRZA_S-1


KeyboardInterrupt: 

### 3. Check sparse files: full copy correctness

In [6]:
import pandas as pd
from pathlib import Path
import glob

NETWORK = "SNOTEL"  # change as needed

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/combine") / NETWORK
SPARSE_DIR   = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean") / NETWORK / "sparse"

# Collect all sparse files
sparse_files = glob.glob(str(SPARSE_DIR / "*.csv"))
print(f"Found {len(sparse_files)} sparse files for {NETWORK}")

problems = []

for f_sparse in sparse_files:
    df_s = pd.read_csv(f_sparse)
    station_name = Path(f_sparse).stem.split("_")[0]  # adapt if station naming differs

    # find original combine file: assume same filename under COMBINE_ROOT/<station>/
    # if your naming differs, adapt this.
    st_dir = COMBINE_ROOT / station_name
    orig_candidates = list(st_dir.glob(Path(f_sparse).name))
    if not orig_candidates:
        problems.append((f_sparse, "Original file not found"))
        continue

    df_o = pd.read_csv(orig_candidates[0])

    # apply same renaming as script did
    rename_map = {}
    if "ts_station_k" in df_o.columns:
        rename_map["ts_station_k"] = "ts_reference"
    if "ts_era5" in df_o.columns:
        rename_map["ts_era5"] = "ts_era5"
    df_o = df_o.rename(columns=rename_map)

    # Compare shapes
    if df_o.shape != df_s.shape:
        problems.append((f_sparse, f"Shape mismatch: orig {df_o.shape}, sparse {df_s.shape}"))
        continue

    # Compare values cell by cell
    if not df_o.equals(df_s):
        problems.append((f_sparse, "Data mismatch between original and sparse copy"))

print(f"\nSparse problems found: {len(problems)}")
for p in problems[:20]:
    print(p)


Found 246 sparse files for SNOTEL

Sparse problems found: 0


### 4. Check for files without any valid reference data

In [7]:
from pathlib import Path
import pandas as pd
import glob
import os

OUT_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean")

dense_empty = []
sparse_empty = []

for net_dir in sorted(OUT_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    dense_dir = net_dir / "dense"
    sparse_dir = net_dir / "sparse"

    # Dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / "*.csv")):
            df = pd.read_csv(f)
            if "ts_reference" in df.columns:
                if df["ts_reference"].notna().sum() == 0:
                    dense_empty.append(f)

    # Sparse
    if sparse_dir.exists():
        for f in glob.glob(str(sparse_dir / "*.csv")):
            df = pd.read_csv(f)
            if "ts_reference" in df.columns:
                if df["ts_reference"].notna().sum() == 0:
                    sparse_empty.append(f)

print("Dense files with no valid ts_reference:", len(dense_empty))
for f in dense_empty[:20]:
    print("  ", f)

print("\nSparse files with no valid ts_reference:", len(sparse_empty))
for f in sparse_empty[:20]:
    print("  ", f)


/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_233915/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read

Dense files with no valid ts_reference: 1
   /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/TWENTE/dense/TWENTE_dense_lat52.5_lon7.0.csv

Sparse files with no valid ts_reference: 0
